In [19]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
%matplotlib widget
from core.CardiacCTdataset import DataLoaderFactory
from core.CNNmodel import *
import logging
from core.Log import *
import pandas as pd
from core.benchmarks import *
setup_loggers5K()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.optim as optim
from sklearn.metrics import roc_auc_score, roc_curve, f1_score
import numpy as np
import copy




def best_threshold(all_labels, all_probs, utility="youden"):
	"""
	Pick a post-hoc decision threshold on validation predictions.
	utility: "youden" (maximize TPR-FPR) or "f1".
	"""
	if utility == "f1":
		# scan unique probabilities for F1
		# (for speed you can sample a subset if very large)
		thr = np.unique(all_probs)
		f1s = [f1_score(all_labels, all_probs >= t) for t in thr]
		idx = int(np.argmax(f1s))
		return float(thr[idx])
	else:
		fpr, tpr, thr = roc_curve(all_labels, all_probs)
		j = tpr - fpr
		idx = int(np.argmax(j))
		return float(thr[idx])  # may be outside [0,1] if degenerate; fine.




def train_OUTER_model(model, train_loader, val_loader, experiment):
	log = logging.getLogger('OUTER_5Ktrain')
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	model.to(device)
	hypers = experiment['hypers']
	ExpID = experiment['ExpID']
	epochs = hypers['Epochs']


	best_th = 0.5
	LR = hypers['LR']
	WD = hypers['WD']
	P = hypers['P']

	optimizer = optim.Adam(model.parameters(), lr= LR, weight_decay=WD)
	scheduler = ReduceLROnPlateau(optimizer, mode='min',
								  patience=2, factor=0.5,
								  threshold=1e-3, threshold_mode='rel',
								  cooldown=0, min_lr=1e-6)
	criterion = nn.BCEWithLogitsLoss()

	best_auc = -np.inf
	best_loss_at_best_auc = np.inf


	no_improve = 0

	val_N=len(val_loader.dataset)
	train_N=len(train_loader.dataset)


	print(f"	↳ Experiment {ExpID} | Training model... ")
	for epoch in range(epochs):
		model.train()
		running_loss = 0.0
		all_labels = []  #y_true
		all_probs = []   #y_pred

		for batch in train_loader:
			#axi = batch["axial_image"].to(device)
			#cor = batch["coronal_image"].to(device)
			img = batch["sagittal_image"].to(device)
			met = batch["meta"].to(device)
			lbl = batch["label"].to(device).unsqueeze(1)

			optimizer.zero_grad(set_to_none=True)

			logits = model(img, met)
			#logits = model(met)
			T_loss = criterion(logits, lbl)

			T_loss.backward()
			optimizer.step()
			all_labels.extend(lbl.detach().cpu())
			all_probs.extend(torch.sigmoid(logits).detach().cpu())

			running_loss += T_loss.item() * lbl.size(0)

		T_loss = running_loss / train_N
		all_labels = torch.cat(all_labels).numpy().reshape(-1)
		all_probs = torch.cat(all_probs).numpy().reshape(-1)
		T_auc = roc_auc_score(all_labels, all_probs)

		model.eval()
		running_loss = 0.0
		all_labels = []  #y_true
		all_probs = []   #y_pred

		with torch.no_grad():
			for batch in val_loader:
				#axi = batch["axial_image"].to(device)
				#cor = batch["coronal_image"].to(device)
				img = batch["sagittal_image"].to(device)
				met = batch["meta"].to(device)
				lbl = batch["label"].to(device).unsqueeze(1)

				logits = model(img, met)
				#logits = model(met)
				V_loss = criterion(logits, lbl)
				running_loss += V_loss.item() * lbl.size(0)

				all_labels.extend(lbl.detach().cpu())
				all_probs.extend(torch.sigmoid(logits).detach().cpu())

		V_loss = running_loss / val_N
		all_labels = torch.cat(all_labels).numpy().reshape(-1)
		all_probs = torch.cat(all_probs).numpy().reshape(-1)
		val_auc = roc_auc_score(all_labels, all_probs)

		scheduler.step(V_loss)

		improved = val_auc > best_auc + 1e-6
		th_star = best_threshold(all_labels, all_probs)

		log.info(f"{experiment['Model']};    {ExpID};    {experiment['OUTER_FOLD']};      {hypers['HPset']};    {epoch:02d};    {T_loss:.4f};    {T_auc:.4f};    {V_loss:.4f};    {val_auc:.4f};    {th_star:.6f};    {optimizer.param_groups[0]['lr']};    {no_improve:02d};    ")

		if improved:
			best_auc = val_auc
			best_loss_at_best_auc = V_loss
			best_epoch = epoch
			best_th = th_star
			no_improve = 0
			best_model_state = copy.deepcopy(model.state_dict())
		else:
			no_improve += 1
		if no_improve >= P: break

	results = {
		"ExpID": ExpID,
		"Model": experiment['Model'],
		'OUTER_FOLD' : experiment['OUTER_FOLD'],
		"HPset": hypers['HPset'],
		"best_val_auc": float(best_auc),
		"best_val_loss": float(best_loss_at_best_auc),
		"best_threshold": float(best_th),
		"best_epoch": int(best_epoch),
		"epochs_ran": int(epoch),
		'trained':True,
		"hypers": hypers}

	return results, best_model_state


def append_experiment_results(item, path="NCV_5_3_folds/OUTER_results.jsonl"):
	with open(path, "a") as f:  # append mode
		f.write(json.dumps(item) + "\n")



In [26]:
full_dl = load_dataset_info(file="NCV_5_3_folds/data_info_5-3NCV.json")
main_dataset = [i for i in full_dl if i["pool"] == 'main']
DL = DataLoaderFactory(main_dataset)


In [7]:
OUTER_CV_parameters = load_from_json("NCV_5_3_folds/OUTER_experiments.json")
df = pd.DataFrame(OUTER_CV_parameters)
df


Loaded NCV_5_3_folds/OUTER_experiments.json.


,ExpID,Model,OUTER_FOLD,hypers,trained,evaluated,HPsetID
0,1,Axial,0,"{'HPset': 4, 'LR': 0.001, 'WD': 0.0001, 'DR': ...",False,False,4
1,2,Axial,1,"{'HPset': 4, 'LR': 0.001, 'WD': 0.0001, 'DR': ...",False,False,4
2,3,Axial,2,"{'HPset': 3, 'LR': 0.001, 'WD': 1e-06, 'DR': 0...",False,False,3
3,4,Axial,3,"{'HPset': 2, 'LR': 0.0003, 'WD': 0.0001, 'DR':...",False,False,2
4,5,Axial,4,"{'HPset': 4, 'LR': 0.001, 'WD': 0.0001, 'DR': ...",False,False,4
5,6,Coronal,0,"{'HPset': 1, 'LR': 0.0003, 'WD': 0.0001, 'DR':...",False,False,1
6,7,Coronal,1,"{'HPset': 1, 'LR': 0.0003, 'WD': 0.0001, 'DR':...",False,False,1
7,8,Coronal,2,"{'HPset': 1, 'LR': 0.0003, 'WD': 0.0001, 'DR':...",False,False,1
8,9,Coronal,3,"{'HPset': 1, 'LR': 0.0003, 'WD': 0.0001, 'DR':...",False,False,1
9,10,Coronal,4,"{'HPset': 1, 'LR': 0.0003, 'WD': 0.0001, 'DR':...",False,False,1


In [35]:
OUTER_CV_parameters = load_from_json("NCV_5_3_folds/OUTER_experiments.json")
filtered = [
	exp for exp in OUTER_CV_parameters
	if exp["Model"] == "Sagittal"
	#and exp["OUTER_FOLD"] == 2       # 0, 1, 2, 3, 4
	#and exp["hypers"]["HPset"] == 4
	and exp["trained"] == False
]
print(f"experiments to do: {len(filtered)}")
log = logging.getLogger('OUTER_5Ktrain')


Loaded NCV_5_3_folds/OUTER_experiments.json.
experiments to do: 5


In [36]:
log.info(f"Model;    ExpID;    OUTER_FOLD;      HPset;    epoch;    T_loss;    T_auc;    V_loss;    val_auc;    th_star;    LR;    no_improve;    ")

for experiment in filtered:
	print(experiment)
	OUT = experiment['OUTER_FOLD']
	model_type = experiment["Model"]
	#train_loader, val_loader, _ = DL.create_outer_loadersMLP(OUT)
	train_loader, val_loader, _ = DL.create_outer_loadersSingleView(OUT)
	DR = experiment['hypers']['DR']
	#model = MetadataMLP(DR)
	model = SingleViewClassifier(DR)
	results, best_model_state = train_OUTER_model(model, train_loader, val_loader, experiment)
	torch.save(best_model_state, f"pth_models_5K/{model_type}_fold_{OUT}.pth")
	append_experiment_results(results, path="NCV_5_3_folds/OUTER_results.jsonl")






{'ExpID': 26, 'Model': 'Sagittal', 'OUTER_FOLD': 0, 'hypers': {'HPset': 1, 'LR': 0.0003, 'WD': 0.0001, 'DR': 0.35, 'TH': 0.5287529428799947, 'P': 5, 'Epochs': 50}, 'trained': False, 'evaluated': False, 'HPsetID': 1}
	↳ Experiment 26 | Training model... 
{'ExpID': 27, 'Model': 'Sagittal', 'OUTER_FOLD': 1, 'hypers': {'HPset': 1, 'LR': 0.0003, 'WD': 0.0001, 'DR': 0.35, 'TH': 0.5321967403093973, 'P': 5, 'Epochs': 50}, 'trained': False, 'evaluated': False, 'HPsetID': 1}
	↳ Experiment 27 | Training model... 
{'ExpID': 28, 'Model': 'Sagittal', 'OUTER_FOLD': 2, 'hypers': {'HPset': 1, 'LR': 0.0003, 'WD': 0.0001, 'DR': 0.35, 'TH': 0.4984995325406392, 'P': 5, 'Epochs': 50}, 'trained': False, 'evaluated': False, 'HPsetID': 1}
	↳ Experiment 28 | Training model... 
{'ExpID': 29, 'Model': 'Sagittal', 'OUTER_FOLD': 3, 'hypers': {'HPset': 1, 'LR': 0.0003, 'WD': 0.0001, 'DR': 0.35, 'TH': 0.5336645245552063, 'P': 5, 'Epochs': 50}, 'trained': False, 'evaluated': False, 'HPsetID': 1}
	↳ Experiment 29 | Tra